# NORIA — REAL talking clips from the REAL image (SadTalker)

Makes **photoreal talking‑Noria videos from your actual renders** — real eye
blinks, real lip‑sync, natural head motion — driven by her voice. Uses
**SadTalker** (single image + audio → talking video), installed inside a
**Python 3.11** environment (Colab is 3.13, which breaks these models).

Honest: these are **rendered offline** (a minute or two each), not live
conversation — but the movement is genuinely real, on the real face.

**Run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

## 0. GPU on?

In [ ]:
!nvidia-smi -L

## 1. Build a Python 3.11 environment + get SadTalker

In [ ]:
!pip -q install uv
!uv python install 3.11
%cd /content
!git clone -q https://github.com/OpenTalker/SadTalker || echo "already cloned"
%cd /content/SadTalker
import os
if not os.path.isdir('.venv'):
    get_ipython().system('uv venv --python 3.11 .venv')  # only create if missing (never prompts)
else:
    print('.venv already exists — keeping it')
print('py3.11 env ready')

## 2. Install into the 3.11 env (this is the step that used to break — now fixed)

In [ ]:
# GPU torch for py3.11
!uv pip install --python .venv/bin/python torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python .venv/bin/python "setuptools<70" wheel
# scientific stack pinned to numpy<2 (SadTalker needs np.float / VisibleDeprecationWarning)
!uv pip install --python .venv/bin/python "numpy==1.26.4" "scipy==1.13.1" "scikit-image==0.22.0" "numba==0.59.1" "llvmlite==0.42.0" "opencv-python==4.9.0.80" "librosa==0.10.2" resampy soundfile imageio imageio-ffmpeg kornia yacs pydub safetensors av tqdm edge-tts
# packages that need the old build system → no build isolation
!uv pip install --python .venv/bin/python --no-build-isolation basicsr facexlib gfpgan "face-alignment==1.3.5"
# re-pin in case a dependency bumped numpy/opencv back up
!uv pip install --python .venv/bin/python "numpy==1.26.4" "opencv-python==4.9.0.80"

## 3. Two compatibility patches (torchvision + numpy)

In [ ]:
%%writefile /content/patch.py
import os
sp = '/content/SadTalker/.venv/lib/python3.11/site-packages'
deg = os.path.join(sp, 'basicsr', 'data', 'degradations.py')  # patch WITHOUT importing basicsr
open(deg, 'w').write(open(deg).read().replace('functional_tensor', 'functional'))
# SadTalker imports the gfpgan enhancer at startup even when unused → make it optional
a = '/content/SadTalker/src/facerender/animate.py'
s = open(a).read()
if 'try:\n    from src.utils.face_enhancer' not in s:
    s = s.replace('from src.utils.face_enhancer import enhancer_generator_with_len, enhancer_list', 'try:\n    from src.utils.face_enhancer import enhancer_generator_with_len, enhancer_list\nexcept Exception as _e:\n    enhancer_generator_with_len = enhancer_list = None')
    open(a, 'w').write(s)
open(os.path.join(sp, "sitecustomize.py"), "w").write(
    "import numpy as np\n"
    "for a,t in [('float',float),('int',int),('bool',bool),('complex',complex),('object',object),('str',str)]:\n"
    "    hasattr(np,a) or setattr(np,a,t)\n"
    "if not hasattr(np,'VisibleDeprecationWarning'): np.VisibleDeprecationWarning=DeprecationWarning\n")
print('patched')

In [ ]:
!.venv/bin/python /content/patch.py
!bash scripts/download_models.sh

## 4. Give Noria a voice line (free neural TTS)
Edit the line to whatever you want her to say.

In [ ]:
NORIA_LINE = "Hello, I'm Noria, your SkyGlobe companion. It's really good to finally meet you."
!.venv/bin/edge-tts --voice en-US-JennyNeural --text "{NORIA_LINE}" --write-media /content/f.mp3
!.venv/bin/edge-tts --voice en-US-GuyNeural   --text "{NORIA_LINE}" --write-media /content/m.mp3
!ffmpeg -y -loglevel error -i /content/f.mp3 -ar 16000 /content/f.wav
!ffmpeg -y -loglevel error -i /content/m.mp3 -ar 16000 /content/m.wav
print('voices ready')

## 5. Get BOTH real Noria faces

In [ ]:
!wget -q -O /content/noria-f.png https://noria-body.onrender.com/assets/noria-f.png
!wget -q -O /content/noria-m.png https://noria-body.onrender.com/assets/noria-m.png
print('faces ready')

## 6. Render the talking clips (real blink + lip-sync) — ~1–3 min each

In [ ]:
%cd /content/SadTalker
!.venv/bin/python inference.py --driven_audio /content/f.wav --source_image /content/noria-f.png --result_dir /content/out_f --still --preprocess full
!.venv/bin/python inference.py --driven_audio /content/m.wav --source_image /content/noria-m.png --result_dir /content/out_m --still --preprocess full
print('done')

## 7. Watch both — and download

In [ ]:
import glob, os
from IPython.display import HTML, display
from base64 import b64encode
def show(tag, d):
    vids = sorted(glob.glob(d + '/**/*.mp4', recursive=True), key=os.path.getmtime)
    if not vids: print('No video for', tag, '— check the render cell output.'); return
    mp4 = vids[-1]
    print(tag, '->', mp4)
    data = b64encode(open(mp4, 'rb').read()).decode()
    display(HTML(f'<b>{tag}</b><br><video width=360 controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))
    try:
        from google.colab import files; files.download(mp4)
    except Exception:
        print('   download from the Files panel:', mp4)
show('Noria-F', '/content/out_f')
show('Noria-M', '/content/out_m')